<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue converts the grouped-by-client model score into a ranked list of pages for human review. Each page is given a reason code based on observable March 2026 search and engagement signals, followed by a suggested review action.
The queue contains 30,557 pages from the held-out test clients. The most common reason code is low_engagement (25,553 pages), followed by high_visibility_low_ctr (2,175), mixed_signals (1,688), and high_visibility_poor_position (1,141).
At the top of the ranking, pages commonly have substantial search visibility but relatively poor average position and low CTR. For example, the highest-ranked page has 25,711 impressions, a CTR of 0.023%, and an average position of 29.8. Its recommended action is to review content relevance, depth and search intent.
These reason codes are intended to make the ranking easier for a human reviewer to interpret. They are not automatic diagnoses that a page needs refreshing.

| Reason code                     | Human-readable reason                                          | Suggested action                                      |
| ------------------------------- | -------------------------------------------------------------- | ----------------------------------------------------- |
| `high_visibility_low_ctr`       | High search visibility but relatively low CTR                  | **Review title/meta/snippet alignment**               |
| `high_visibility_poor_position` | High search visibility but relatively poor average position    | **Review content relevance, depth and search intent** |
| `low_engagement`                | Search/traffic signals exist but engagement is relatively weak | **Review content usefulness and page experience**     |
| `mixed_signals`                 | Multiple signals indicate potential opportunity                | **Manual content review**                             |
| `no_clear_issue`                | No strong refresh signal from the measured features            | **Lower priority / monitor**                          |


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

march_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

        BOOL_OR(gsc_data_available) AS gsc_data_available,
        BOOL_OR(ga4_data_available) AS ga4_data_available

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

import numpy as np

march_df["gsc_ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

march_df["engagement_rate"] = (
    march_df["ga4_engaged_sessions"] /
    march_df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

features = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

X = march_df[features].copy()

# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values
X = X.fillna(0)

# 1. Prepare March dataset

proxy_df = march_df.copy()

print("Rows:", len(proxy_df))

feature_cols = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]


import numpy as np

proxy_df["log_impressions"] = np.log1p(
    proxy_df["gsc_impressions"]
)

proxy_df["log_sessions"] = np.log1p(
    proxy_df["ga4_sessions"]
)

proxy_df["log_engagement"] = np.log1p(
    proxy_df["engagement_rate"]
)

model_features = [
    "log_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "log_sessions",
    "log_engagement"
]

print("Model features:", model_features)

# 3. Create March-only proxy target

proxy_df["impression_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_impressions"]
    .rank(pct=True)
)

proxy_df["ctr_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_ctr"]
    .rank(pct=True)
)

proxy_df["position_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_avg_position"]
    .rank(pct=True)
)

proxy_df["engagement_percentile"] = (
    proxy_df.groupby("client_hash_id")["engagement_rate"]
    .rank(pct=True)
)

# A page is "review-worthy" if:
# - it has relatively high visibility within its client
# - AND it has at least one sign of underperformance

proxy_df["target"] = (
    (proxy_df["impression_percentile"] >= 0.75)
    &
    (
        (proxy_df["ctr_percentile"] <= 0.25)
        |
        (proxy_df["position_percentile"] >= 0.75)
        |
        (proxy_df["engagement_percentile"] <= 0.25)
    )
).astype(int)


# 4. Define X, y and groups

X = proxy_df[model_features].copy()

# Missing numeric values are filled with 0.
X = X.fillna(0)

y = proxy_df["target"]

groups = proxy_df["client_hash_id"]


# 5. Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# 6. Train Random Forest classifier

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


# 7. Generate ML scores

test_results = proxy_df.iloc[test_idx].copy()

test_results["ml_score"] = rf.predict_proba(
    X_test
)[:, 1]

# 8. Rank pages using ML score

test_results = test_results.sort_values(
    "ml_score",
    ascending=False
).reset_index(drop=True)

test_results["ml_rank"] = (
    np.arange(len(test_results)) + 1
)

ml_top20 = test_results.head(20)




march_df["impression_percentile"] = (
    march_df.groupby("client_hash_id")["gsc_impressions"]
    .rank(pct=True)
)

march_df["ctr_percentile"] = (
    march_df.groupby("client_hash_id")["gsc_ctr"]
    .rank(pct=True)
)

march_df["position_percentile"] = (
    march_df.groupby("client_hash_id")["gsc_avg_position"]
    .rank(pct=True)
)

march_df["engagement_percentile"] = (
    march_df.groupby("client_hash_id")["engagement_rate"]
    .rank(pct=True)
)

march_df["target"] = (
    (march_df["impression_percentile"] >= 0.75)
    &
    (
        (march_df["ctr_percentile"] <= 0.25)
        |
        (march_df["position_percentile"] >= 0.75)
        |
        (march_df["engagement_percentile"] <= 0.25)
    )
).astype(int)

print("Target distribution:")
print(march_df["target"].value_counts())

print("\nTarget rate:")
print(march_df["target"].mean())



from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

march_df["log_impressions"] = np.log1p(
    march_df["gsc_impressions"]
)

march_df["log_sessions"] = np.log1p(
    march_df["ga4_sessions"]
)

march_df["log_engagement"] = np.log1p(
    march_df["engagement_rate"]
)

features = [
    "log_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "log_sessions",
    "log_engagement"
]

X = march_df[features]
y = march_df["target"]
groups = march_df["client_hash_id"]


def make_model():
    return RandomForestClassifier(
        class_weight="balanced",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    )


# -------------------------
# BEFORE: row-level random split
# -------------------------

X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_before = make_model()
model_before.fit(X_train_before, y_train_before)

pred_before = model_before.predict_proba(X_test_before)[:, 1]

roc_before = roc_auc_score(y_test_before, pred_before)
ap_before = average_precision_score(y_test_before, pred_before)


# -------------------------
# AFTER: grouped-by-client split
# -------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_after = X.iloc[train_idx]
X_test_after = X.iloc[test_idx]

y_train_after = y.iloc[train_idx]
y_test_after = y.iloc[test_idx]

groups_train_after = groups.iloc[train_idx]
groups_test_after = groups.iloc[test_idx]

model_after = make_model()
model_after.fit(X_train_after, y_train_after)

pred_after = model_after.predict_proba(X_test_after)[:, 1]

roc_after = roc_auc_score(y_test_after, pred_after)
ap_after = average_precision_score(y_test_after, pred_after)


# -------------------------
# Comparison
# -------------------------

comparison = pd.DataFrame({
    "split": [
        "Before: row-level random",
        "After: grouped by client"
    ],
    "train_rows": [
        len(X_train_before),
        len(X_train_after)
    ],
    "test_rows": [
        len(X_test_before),
        len(X_test_after)
    ],
    "train_clients": [
        march_df.loc[X_train_before.index, "client_hash_id"].nunique(),
        groups_train_after.nunique()
    ],
    "test_clients": [
        march_df.loc[X_test_before.index, "client_hash_id"].nunique(),
        groups_test_after.nunique()
    ],
    "roc_auc": [
        roc_before,
        roc_after
    ],
    "average_precision": [
        ap_before,
        ap_after
    ]
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437
Model features: ['log_impressions', 'gsc_ctr', 'gsc_avg_position', 'log_sessions', 'log_engagement']
Target distribution:
target
0    319373
1     12064
Name: count, dtype: int64

Target rate:
0.03639907433388547


In [2]:
# 1. Ranked actions + reason codes

queue = test_results.copy()

# Calculate thresholds from the grouped-by-client test set
high_visibility_threshold = queue["gsc_impressions"].quantile(0.75)
low_ctr_threshold = queue["gsc_ctr"].quantile(0.25)
poor_position_threshold = queue["gsc_avg_position"].quantile(0.75)
low_engagement_threshold = queue["engagement_rate"].quantile(0.25)


def get_reason_code(row):

    high_visibility = (
        row["gsc_impressions"] >= high_visibility_threshold
    )

    low_ctr = (
        row["gsc_ctr"] <= low_ctr_threshold
    )

    poor_position = (
        row["gsc_avg_position"] >= poor_position_threshold
    )

    low_engagement = (
        row["engagement_rate"] <= low_engagement_threshold
    )

    if high_visibility and low_ctr:
        return "high_visibility_low_ctr"

    elif high_visibility and poor_position:
        return "high_visibility_poor_position"

    elif low_engagement:
        return "low_engagement"

    else:
        return "mixed_signals"


queue["reason_code"] = queue.apply(
    get_reason_code,
    axis=1
)


action_map = {
    "high_visibility_low_ctr":
        "Review title/meta/snippet alignment",

    "high_visibility_poor_position":
        "Review content relevance, depth and search intent",

    "low_engagement":
        "Review content usefulness and page experience",

    "mixed_signals":
        "Manual content review"
}


queue["recommended_action"] = (
    queue["reason_code"].map(action_map)
)


ranked_queue = queue[
    [
        "ml_rank",
        "client_hash_id",
        "content_hash_id",
        "ml_score",
        "reason_code",
        "recommended_action",
        "gsc_impressions",
        "gsc_ctr",
        "gsc_avg_position",
        "ga4_sessions",
        "engagement_rate"
    ]
].sort_values("ml_rank")


print("Queue rows:", len(ranked_queue))
print("\nReason codes:")
print(ranked_queue["reason_code"].value_counts())

print("\nTop 20 actions:")
display(ranked_queue.head(20))

Queue rows: 30557

Reason codes:
reason_code
low_engagement                   25553
high_visibility_low_ctr           2175
mixed_signals                     1688
high_visibility_poor_position     1141
Name: count, dtype: int64

Top 20 actions:


,ml_rank,client_hash_id,content_hash_id,ml_score,reason_code,recommended_action,gsc_impressions,gsc_ctr,gsc_avg_position,ga4_sessions,engagement_rate
0,1,client_e547b89c05043229,content_e986eb3ab70fcee6,0.998408,high_visibility_poor_position,"Review content relevance, depth and search intent",25711.0,0.000233,29.844970,28.0,0.035714
1,2,client_e547b89c05043229,content_cd33c3f6ef059886,0.997157,high_visibility_poor_position,"Review content relevance, depth and search intent",11420.0,0.000263,38.534245,3.0,0.000000
2,3,client_e547b89c05043229,content_fecf82eefd6d232e,0.997137,high_visibility_poor_position,"Review content relevance, depth and search intent",8362.0,0.000239,33.501989,3.0,0.000000
3,4,client_e547b89c05043229,content_a49d621d476664e2,0.997002,high_visibility_poor_position,"Review content relevance, depth and search intent",12665.0,0.000237,56.475395,6.0,0.000000
4,5,client_e547b89c05043229,content_73325ca66a8a63cc,0.996908,high_visibility_poor_position,"Review content relevance, depth and search intent",45653.0,0.001336,37.470108,47.0,0.063830
5,6,client_e547b89c05043229,content_805c0c2e555a8057,0.996881,high_visibility_poor_position,"Review content relevance, depth and search intent",12511.0,0.000240,50.673126,3.0,0.000000
6,7,client_e547b89c05043229,content_10e8f76ed8c5c392,0.996526,high_visibility_poor_position,"Review content relevance, depth and search intent",18398.0,0.000054,48.569375,5.0,0.000000
7,8,client_e547b89c05043229,content_e3b2a512ba3b320e,0.996391,high_visibility_poor_position,"Review content relevance, depth and search intent",9815.0,0.000408,39.227902,7.0,0.000000
8,9,client_e547b89c05043229,content_23b8547eb4f16144,0.996366,high_visibility_poor_position,"Review content relevance, depth and search intent",5998.0,0.000333,37.335395,3.0,0.000000
9,10,client_e547b89c05043229,content_413865fe64925077,0.996203,high_visibility_poor_position,"Review content relevance, depth and search intent",6327.0,0.000316,31.406315,4.0,0.000000


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended use

The intended user is a content or SEO team deciding which pages to review first when review time is limited.

The model provides a directional ranking of pages based on observable March 2026 search and engagement signals. The ranking can be used to prioritise human review and to provide an initial reason for why a page was surfaced, such as high visibility with low CTR, poor average position, or low engagement.

The output is therefore decision-support rather than an automated refresh decision. A reviewer should use the ranking alongside their knowledge of the page, search intent, competition and business context.

### Limits

The model should not be interpreted as predicting which pages will definitely benefit from a content refresh. The proxy target was constructed from the same March 2026 signals used as model features, so the evaluation measures how well the model reproduces this proxy rather than whether a refresh would actually improve future performance.

The analysis is also restricted to March 2026, with the final evaluation using held-out clients. It therefore does not establish performance across future months or prove that the ranking generalises to all clients.

Poor search position, low CTR or low engagement can have causes other than outdated content, including competition, search intent or other external factors. For this reason, the model score and reason code should be treated as signals for investigation, not as automatic diagnoses.

The model should not automatically publish, rewrite or remove content. A human reviewer remains responsible for deciding whether a page actually needs a refresh and what action is appropriate.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Before acting on a page surfaced by the model, a reviewer should check:

1. **Search intent:** Does the page actually match the intent behind the queries generating its impressions?
2. **Content quality and relevance:** Is the content outdated, incomplete, inaccurate or otherwise in need of improvement?
3. **Competition:** Is the poor average position likely to reflect strong competition rather than a content problem?
4. **CTR context:** Is the low CTR plausibly related to the title, meta description or search-result presentation?
5. **Engagement context:** Does low engagement indicate a genuine content or page-experience issue, or could it be explained by the type of page or traffic it receives?
6. **Business context:** Is the page important enough to justify the time and resources required for a refresh?
7. **Reason-code evidence:** Does the page's observed data actually support the reason code assigned by the queue?

The reviewer should record a final decision such as **refresh**, **investigate further**, or **do not refresh**. The model score should not determine the final decision by itself.

### No-go list

The following should not be automated from this model:

- Automatically rewriting or publishing page content.
- Automatically deleting, redirecting or removing pages.
- Automatically deciding that a page is outdated or low quality.
- Automatically treating poor rankings as evidence that a content refresh will improve performance.
- Automatically changing titles or metadata without human review.
- Automatically prioritising pages solely because they have a high model score.
- Using the score as evidence of causal impact or guaranteed future performance.

The model is therefore a **review-prioritisation tool**, not an autonomous content optimisation system. Its role stops when a human needs to determine why a page is performing as it is and whether a particular action is appropriate.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


The recommendations should be reviewed if the data or the model's behaviour changes enough that the March 2026 ranking may no longer represent current content performance.

### Monitoring

The following should be monitored over time:

- **Data availability:** Check for substantial changes in the availability of GSC or GA4 data, since missingness is common in the March dataset and is not necessarily random.
- **Feature distributions:** Monitor changes in impressions, CTR, average position, sessions and engagement compared with the data used to train the model.
- **Score distribution:** Check whether the proportion of pages receiving very high or very low model scores changes substantially.
- **Reason-code distribution:** Monitor whether the balance of `low_engagement`, `high_visibility_low_ctr`, `high_visibility_poor_position` and `mixed_signals` changes substantially.
- **Model performance:** When a suitable future evaluation target becomes available, check whether ranking performance deteriorates on new data.

### Retrain or review triggers

The model should be reconsidered or retrained when:

1. The underlying search or engagement distributions change substantially.
2. Data collection or availability changes, particularly for GSC or GA4.
3. The model begins producing a substantially different score or reason-code distribution.
4. A future evaluation shows that the model's ranking performance has deteriorated.
5. The content-review workflow or definition of a useful refresh opportunity changes.
6. A better future-looking outcome becomes available, such as observed performance after a refresh. In that case, the current March proxy target should be reconsidered rather than simply retraining on the same proxy.

Until these checks are performed, the model should not be assumed to remain valid for new periods or clients. Monitoring is therefore a trigger for review, not a guarantee that retraining will fix the underlying issue.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported to `work/outputs/w07_ranked_action_queue.csv` so that the paper can reuse the same ranked recommendations produced by the notebook.

The exported queue contains the model rank and score, page and client identifiers, reason code, recommended action, and the underlying search and engagement signals used to support the recommendation.

Any figures used in the paper will also be saved to `work/outputs/` and, where appropriate, copied to `work/figures/` for inclusion in the final report.

The queue is an output for analysis and human review; it is not treated as an automatically executable content-change list.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue for reuse in the paper
queue_path = output_dir / "w07_ranked_action_queue.csv"
ranked_queue.to_csv(queue_path, index=False)

print(f"Saved queue to: {queue_path}")
print(f"Rows exported: {len(ranked_queue):,}"

Saved queue to: ../outputs/w07_ranked_action_queue.csv
Rows exported: 30,557


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.